# Grad-CAM — CXR 폐렴 3분류 (v4 / batch16, test macro-F1 0.866)

SageMaker JupyterLab에서 실행. 모델/데이터는 **S3에서 직접** 읽는다 (노트북 IAM 롤 사용, `--profile` 불필요).

**이 노트북이 하는 일**
1. 최종 모델(ResNet18, v4/batch16) S3에서 로드
2. 학습 때와 **동일한 eval 전처리**(증강 없음)로 추론
3. Grad-CAM 직접 구현 (`layer4[-1]` feature map + gradient)
4. **실시간 단건**: 샘플 CXR → 분류 확률 + heatmap + overlay 표시 (← 오늘의 메인)
5. **배치 저장 함수**: 나중에 1000장 한 번에 `education-game/` 구조로 S3 저장 (함수만 준비, 지금은 실행 X)

**클래스 순서 (고정)**: `0=NORMAL(정상)`, `1=BACTERIAL(세균성)`, `2=VIRAL(바이러스성)`

## CELL 0 — 설정 (S3 경로 / 상수)
학습 당시 config.json과 동일하게 맞춘 값. 바꿀 일 있으면 여기만 고친다.

In [ ]:
# ---- S3 경로 ----
# 최종 모델 (v4/batch16, test macro-F1 0.866)
MODEL_BUCKET = "aws-say3-student-dataset"
MODEL_KEY = "students/aws-say3-11/P1/model_dev/v4/run_20261014_0534_batch16/best_model.pth"

# 이미지 데이터 버킷 (읽기 전용)
DATA_BUCKET = "aws-say3-dataset1"
DATA_PREFIX = "pneumonia-xray/chest_xray/"  # relative_path 앞에 붙는 접두사

# 교육게임 저장 위치 (배치 저장 시). 하위에 images/ gradcam/ overlay/ questions.csv 생성
OUT_BUCKET = "aws-say3-student-dataset"
OUT_PREFIX = "students/aws-say3-11/P1/education_gradcam/"

# ---- 라벨 (학습 때 순서 그대로! 절대 바꾸지 말 것) ----
CLASS_NAMES = ["NORMAL", "BACTERIAL_PNEUMONIA", "VIRAL_PNEUMONIA"]
CLASS_NAMES_KR = ["정상", "세균성 폐렴", "바이러스성 폐렴"]
NUM_CLASSES = 3

# ---- 이미지 전처리 (학습 eval transform과 동일, 증강 없음) ----
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# CSV (SageMaker 노트북에 올려둔 전체 라벨 마스터). CELL 8에서 split=='test'만 거른다.
CSV_PATH = "pneumonia_labels.csv"

print("설정 로드 완료")
print("모델:", f"s3://{MODEL_BUCKET}/{MODEL_KEY}")

## CELL 1 — import & 디바이스

In [ ]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T

import boto3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device)

s3 = boto3.client("s3")  # SageMaker 노트북 IAM 롤 자격증명 자동 사용

## CELL 2 — 모델 로드 (S3 → ResNet18)
학습 때 구조와 동일: ImageNet ResNet18의 fc를 512→3으로 교체 후 가중치 로드.

In [ ]:
def build_model():
    """학습 때와 동일한 구조. 가중치는 뒤에서 덮어쓰므로 pretrained 불필요."""
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def load_model_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    ckpt = torch.load(io.BytesIO(obj["Body"].read()), map_location=device, weights_only=False)

    # 이 체크포인트는 dict로 저장됨: {'model_state', 'class_names', 'img_size', 'mean', 'std'}
    # state_dict가 어느 키에 들었는지 자동 탐색 (model_state / model / state_dict / 통째)
    state = ckpt
    if isinstance(ckpt, dict):
        for k in ("model_state", "model", "state_dict"):
            if k in ckpt and isinstance(ckpt[k], dict):
                state = ckpt[k]
                break
        # 체크포인트에 저장된 메타로 클래스 순서/전처리 교차확인
        if "class_names" in ckpt:
            print("[info] 체크포인트 class_names:", ckpt["class_names"])
            assert list(ckpt["class_names"]) == CLASS_NAMES, \
                f"클래스 순서 불일치! ckpt={ckpt['class_names']} vs 노트북={CLASS_NAMES}"
        for mk in ("img_size", "mean", "std"):
            if mk in ckpt:
                print(f"[info] 체크포인트 {mk}:", ckpt[mk])

    model = build_model()
    missing, unexpected = model.load_state_dict(state, strict=False)
    # ★ 조용한 로드 실패 방지: 핵심 가중치가 안 들어오면 즉시 멈춘다
    if missing:
        raise RuntimeError(
            f"가중치 로드 실패! {len(missing)}개 키가 안 채워짐 (예: {missing[:3]}). "
            "state_dict 추출 키를 확인하세요."
        )
    if unexpected:
        print("[info] 모델에 없는 여분 키(무시됨):", unexpected)
    model.to(device).eval()
    print("[OK] 학습된 가중치 로드 성공 (missing 0개)")
    return model


model = load_model_from_s3(MODEL_BUCKET, MODEL_KEY)
print("모델 로드 완료. fc:", model.fc)

## CELL 3 — 전처리 (학습 eval transform과 동일)
흑백 X선 → RGB 3채널, Resize 224, ImageNet 정규화. **증강 없음**(추론용).

In [ ]:
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


def load_image_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return Image.open(io.BytesIO(obj["Body"].read())).convert("RGB")


def load_image(relative_path=None, s3_uri=None, local_path=None):
    """이미지 한 장을 PIL(RGB)로 로드. S3/로컬 모두 지원."""
    if local_path and os.path.exists(local_path):
        return Image.open(local_path).convert("RGB")
    if s3_uri:
        assert s3_uri.startswith("s3://")
        b, k = s3_uri[5:].split("/", 1)
        return load_image_from_s3(b, k)
    if relative_path:
        return load_image_from_s3(DATA_BUCKET, DATA_PREFIX + relative_path)
    raise ValueError("경로를 하나는 줘야 함")


def preprocess(pil_img):
    """PIL → 모델 입력 텐서 (1,3,224,224)."""
    return eval_tf(pil_img).unsqueeze(0).to(device)

print("전처리 준비 완료")

## CELL 4 — Grad-CAM 직접 구현
ResNet18의 마지막 conv 블록 `layer4[-1]`에 forward/backward hook을 걸어
**feature map**과 **gradient**를 잡고, gradient 공간 평균을 채널 가중치로 써서 CAM을 만든다.

In [ ]:
class GradCAM:
    """ResNet18 Grad-CAM. target_layer의 activation과 gradient로 히트맵 생성."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_activation)
        # full backward hook: 이 레이어 출력에 대한 gradient를 받는다
        self._bwd = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def remove(self):
        self._fwd.remove()
        self._bwd.remove()

    def __call__(self, input_tensor, class_idx=None):
        """입력 1장에 대한 (cam[H,W] 0~1, pred_idx, probs[np]) 반환.
        class_idx=None이면 예측 클래스 기준으로 CAM 생성.
        """
        self.model.zero_grad()
        logits = self.model(input_tensor)            # (1,3)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        # 타깃 클래스 점수에 대해 역전파
        score = logits[0, class_idx]
        score.backward(retain_graph=False)

        grads = self.gradients[0]        # (C,h,w)
        acts = self.activations[0]       # (C,h,w)
        weights = grads.mean(dim=(1, 2))  # (C,)  채널별 gradient 공간평균
        cam = (weights[:, None, None] * acts).sum(dim=0)  # (h,w)
        cam = F.relu(cam)                 # 양의 기여만
        # 입력 크기로 업샘플 후 0~1 정규화
        cam = cam[None, None]
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, class_idx, probs


# 타깃 레이어: ResNet18 마지막 conv 블록
target_layer = model.layer4[-1]
gradcam = GradCAM(model, target_layer)
print("Grad-CAM 준비 완료. target =", target_layer.__class__.__name__)

## CELL 5 — 시각화 유틸 (heatmap + overlay)
CAM을 컬러 heatmap으로, 그리고 원본 위에 반투명 overlay로. 둘 다 numpy 배열로 반환.

In [ ]:
from matplotlib import cm as mpl_cm


def cam_to_heatmap(cam, colormap="jet"):
    """cam(H,W 0~1) → RGB uint8 heatmap (H,W,3)."""
    cmap = mpl_cm.get_cmap(colormap)
    heat = cmap(cam)[..., :3]  # RGBA→RGB, 0~1
    return (heat * 255).astype(np.uint8)


def make_overlay(pil_img, cam, alpha=0.4, colormap="jet"):
    """원본(224로 리사이즈) 위에 heatmap을 alpha로 합성 → RGB uint8 (H,W,3)."""
    base = pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")
    base = np.asarray(base).astype(np.float32)
    heat = cam_to_heatmap(cam, colormap).astype(np.float32)
    over = (1 - alpha) * base + alpha * heat
    return np.clip(over, 0, 255).astype(np.uint8)


def make_overlay_thresh(pil_img, cam, thresh=0.7, alpha=0.6, colormap="jet"):
    """★ CAM이 thresh 이상인 '가장 빨간(핵심) 영역'만 색칠, 나머지는 원본 그대로.
    게임에서 '포인트로 딱 찍는' 용도라 좁고 또렷하게 남기는 게 좋다.
    - thresh: 0~1. 높일수록 더 좁은 핵심만 남음 (게임용 0.7 권장, 0.75~0.8이면 거의 점).
    - alpha : 색칠 영역의 진하기.
    """
    base = pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")
    base = np.asarray(base).astype(np.float32)
    heat = cam_to_heatmap(cam, colormap).astype(np.float32)
    mask = (cam >= thresh).astype(np.float32)[..., None]   # 핵심 영역만 1
    # 핵심 영역: 원본*(1-alpha) + heat*alpha, 그 외: 원본 그대로
    over = base * (1 - alpha * mask) + heat * (alpha * mask)
    return np.clip(over, 0, 255).astype(np.uint8)


def cam_peak_point(cam):
    """★ 게임 정답 좌표: CAM이 가장 뜨거운 '한 점'(피크)의 (x, y) 픽셀 좌표.
    224x224 기준. 게임에서 사용자가 찍은 점과 이 점의 거리로 채점하면 된다.
    반환: (px, py) 0~223
    """
    yy, xx = np.unravel_index(int(np.argmax(cam)), cam.shape)
    return int(xx), int(yy)


def scoring_params(cam, thresh=0.7, r_full=12):
    """★ 게임 채점용 파라미터를 계산 (원 근사).
    - peak        : (px, py) 만점존 중심 = CAM 피크
    - r_full      : 만점존 반지름 (이 안을 찍으면 100점). 기본 12px(224 기준).
    - r_cam       : CAM 영역(>=thresh)을 원으로 근사한 반지름 = sqrt(area/pi).
                    피크에서 이 반지름 안이면 'CAM 영역 안'으로 간주.
    반환 dict (questions.csv에 그대로 저장).
    """
    px, py = cam_peak_point(cam)
    area = int((cam >= thresh).sum())
    r_cam = float(np.sqrt(area / np.pi)) if area > 0 else float(r_full)
    r_cam = max(r_cam, r_full + 1)  # 만점존보다는 크게 보장
    return {"peak_x": px, "peak_y": py, "r_full": float(r_full), "r_cam": round(r_cam, 2)}


def score_click(click_x, click_y, peak_x, peak_y, r_full, r_cam,
                outside_penalty=3.0, min_score=0):
    """★ 학생 클릭 → 점수(0~100). 게임이 이 함수를 그대로 쓰면 된다.
    3단 구조:
      1) 만점존(d<=r_full)        → 100
      2) CAM 영역 안(r_full<d<=r_cam) → 100에서 거리비례 완만 감점 (경계에서 50점)
      3) CAM 영역 밖(d>r_cam)      → 50에서 더 가파르게 감점 (outside_penalty 배속)
    outside_penalty: 영역 밖 감점 가속 배수 (클수록 밖을 더 크게 깎음).
    """
    d = ((click_x - peak_x) ** 2 + (click_y - peak_y) ** 2) ** 0.5
    if d <= r_full:
        return 100.0
    if d <= r_cam:
        # r_full에서 100 → r_cam 경계에서 50 으로 선형 감소
        frac = (d - r_full) / max(r_cam - r_full, 1e-6)
        return round(100 - 50 * frac, 1)
    # 영역 밖: 경계(50점)에서 거리당 더 가파르게. 거리 단위는 r_cam 대비 비율.
    over = (d - r_cam) / max(r_cam, 1e-6)
    return round(max(min_score, 50 - 50 * outside_penalty * over), 1)


def draw_peak(img_arr, point, r=6):
    """이미지 배열 위에 피크 지점을 원으로 표시 (확인용). img_arr: (H,W,3) uint8."""
    out = img_arr.copy()
    px, py = point
    H, W = out.shape[:2]
    yy, xx = np.ogrid[:H, :W]
    ring = (np.abs((xx - px) ** 2 + (yy - py) ** 2 - r ** 2) <= r * 2)
    out[ring] = [0, 255, 0]  # 초록 원
    return out


def make_mask_only(pil_img, cam, thresh=0.7, dim=0.35):
    """(선택) 핵심 영역만 원본 컬러로 밝게, 나머지는 어둡게 깔아 핵심을 '스포트라이트'.
    색(jet) 없이 원본 위에 핵심만 강조하고 싶을 때.
    """
    base = pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")
    base = np.asarray(base).astype(np.float32)
    mask = (cam >= thresh).astype(np.float32)[..., None]
    out = base * (mask + (1 - mask) * dim)   # 바깥은 dim배로 어둡게
    return np.clip(out, 0, 255).astype(np.uint8)


def predict_and_cam(pil_img, class_idx=None, thresh=0.7):
    """한 장 → (probs, pred_idx, cam, heatmap, overlay, overlay_thresh, peak).
    overlay       : 전체 heatmap 합성 (기존)
    overlay_thresh: 핵심(빨간) 영역만 강조
    peak          : (x, y) 가장 뜨거운 한 점 — 게임 정답 좌표 (224 기준)
    """
    x = preprocess(pil_img)
    cam, pred_idx, probs = gradcam(x, class_idx=class_idx)
    heatmap = cam_to_heatmap(cam)
    overlay = make_overlay(pil_img, cam)
    overlay_thresh = make_overlay_thresh(pil_img, cam, thresh=thresh)
    peak = cam_peak_point(cam)
    return probs, pred_idx, cam, heatmap, overlay, overlay_thresh, peak

print("시각화 유틸 준비 완료 (threshold 핵심영역 강조 + 피크 좌표 포함)")

## CELL 6 — ★ 실시간 단건 Grad-CAM (오늘의 메인)
test 이미지 한 장을 골라 분류 확률 + 원본/heatmap/overlay를 한 줄로 표시.
`REL` 값만 바꿔가며 여러 장 확인해보면 된다.

In [ ]:
# 확인할 이미지 하나 선택 (relative_path). test 세트에서 아무거나.
# 예시 3장 — 정상 / 세균성 / 바이러스성
SAMPLES = {
    "정상(NORMAL)": "test/NORMAL/IM-0001-0001.jpeg",
    "세균성(BACTERIA)": "test/PNEUMONIA/person100_bacteria_475.jpeg",
    "바이러스성(VIRUS)": "test/PNEUMONIA/person1_virus_6.jpeg",
}


def show_one(relative_path, title="", thresh=0.7):
    """thresh: 핵심영역 문턱값(0~1). 높일수록 더 좁은 '빨간 핵심'만 남음 (게임용 0.7~0.8)."""
    pil = load_image(relative_path=relative_path)
    probs, pred_idx, cam, heatmap, overlay, overlay_thresh, peak = predict_and_cam(pil, thresh=thresh)

    prob_str = "  ".join(
        f"{CLASS_NAMES_KR[i]} {probs[i]*100:5.1f}%" for i in range(NUM_CLASSES)
    )
    print(f"[{title}] {relative_path}")
    print("  예측:", CLASS_NAMES_KR[pred_idx], "|", prob_str)

    fig, ax = plt.subplots(1, 4, figsize=(17, 4.5))
    ax[0].imshow(pil.resize((IMG_SIZE, IMG_SIZE)), cmap="gray")
    ax[0].set_title("원본 CXR")
    ax[1].imshow(heatmap)
    ax[1].set_title("Grad-CAM heatmap")
    ax[2].imshow(overlay)
    ax[2].set_title("overlay (전체)")
    ax[3].imshow(overlay_thresh)
    ax[3].set_title(f"핵심영역만 (cam>={thresh})")
    for a in ax:
        a.axis("off")
    fig.suptitle(f"{title} → 예측: {CLASS_NAMES_KR[pred_idx]} ({probs[pred_idx]*100:.1f}%)", fontsize=13)
    plt.tight_layout()
    plt.show()


for title, rel in SAMPLES.items():
    try:
        show_one(rel, title)
    except Exception as e:
        print(f"[skip] {rel}: {e}")

### (선택) 내가 고른 이미지 한 장만 빠르게 보기
`REL`에 경로만 넣고 실행. test CSV에 있는 아무 `relative_path`나 가능.

In [ ]:
REL = "test/PNEUMONIA/person100_bacteria_475.jpeg"
show_one(REL, title="단건 확인")

## CELL 7 — (준비만) 배치 저장 함수: education-game 구조로 S3 업로드
나중에 1000장 한 번에 저장할 때 쓸 함수. **지금은 정의만 하고 실행하지 않는다.**

저장 구조:
```
education-game/
├── images/   0001.png, 0002.png, ...   (원본 CXR, 224 리사이즈)
├── gradcam/  0001.png, 0002.png, ...   (overlay 이미지)
└── questions.csv  (id, relative_path, 정답label, 예측label, 확률 ...)
```

In [ ]:
# 교육게임 gradcam 이미지의 핵심영역 문턱값. 게임에서 '점으로 찍는' 용도라 높게.
# CELL 6에서 보기 좋았던 값으로 맞출 것 (0.7 기본, 0.75~0.8이면 거의 점 하나).
GAME_THRESH = 0.7
GAME_R_FULL = 12   # 만점존 반지름(px, 224 기준). 클릭이 피크에서 이 안이면 100점.


def _png_bytes(arr_or_pil):
    """numpy(H,W,3) 또는 PIL → PNG bytes."""
    img = arr_or_pil if isinstance(arr_or_pil, Image.Image) else Image.fromarray(arr_or_pil)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf


def sample_by_ratio(df, n_total, ratio=None, seed=42):
    """클래스(label 0/1/2) 비율을 맞춰 n_total개를 뽑는다.
    - ratio=None  → 세 클래스 '균등' (교육용 권장). 예: 100 → 34/33/33
    - ratio=[a,b,c] → 지정 비율 (합이 1이 아니어도 자동 정규화). 예: test분포 [234,242,148]
    각 클래스 안에서는 섞어서(shuffle) 뽑으므로 매번 같은 seed면 동일 결과.
    클래스에 데이터가 모자라면 있는 만큼만 가져온다.
    """
    import numpy as np
    labels = [0, 1, 2]
    if ratio is None:
        ratio = [1, 1, 1]
    ratio = np.array(ratio, dtype=float)
    ratio = ratio / ratio.sum()
    # 클래스별 목표 개수 (반올림 후 합을 n_total에 맞춤)
    want = [int(round(n_total * r)) for r in ratio]
    diff = n_total - sum(want)
    want[int(np.argmax(ratio))] += diff  # 오차는 비중 큰 클래스에 흡수

    picked = []
    print("클래스별 샘플링 목표:")
    for lab, w in zip(labels, want):
        sub = df[df["label"] == lab]
        take = min(w, len(sub))
        if take < w:
            print(f"  [주의] {CLASS_NAMES_KR[lab]}: 요청 {w}장 < 보유 {len(sub)}장 → {take}장만")
        picked.append(sub.sample(n=take, random_state=seed))
        print(f"  {CLASS_NAMES_KR[lab]:12s}: {take}장")
    out = pd.concat(picked).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    print(f"→ 총 {len(out)}장 선택 (섞음)")
    return out


def build_education_game(df, per_class=None, start_id=1, save_heatmap_too=False,
                         correct_only=True):
    """df를 순회하며 '모델이 맞춘' 이미지만 골라 images/ + gradcam/ + overlay/ + questions.csv 저장.

    교육용이므로 기본적으로 **정답 이미지만** 사용한다 (correct_only=True).
    - per_class: {0:n0, 1:n1, 2:n2} 클래스별 목표 개수. 다 채우면 조기 종료.
                 None이면 df에 있는 '맞춘 것 전부' 저장.
    - df는 넉넉히 넘길 것 (틀린 게 섞여 걸러지므로). 보통 test 전체(624장)를 섞어서 넘긴다.
    df 필요 컬럼: relative_path, label
    실행하면 실제로 S3에 업로드하므로, 확인 끝나고 의도적으로 호출할 것.
    """
    rows = []
    got = {0: 0, 1: 0, 2: 0}        # 클래스별 담은 개수
    skipped_wrong = 0               # 틀려서 버린 개수
    sub = df.reset_index(drop=True)
    next_id = start_id
    for _, r in sub.iterrows():
        # 목표 다 채웠으면 종료
        if per_class is not None and all(got[c] >= per_class.get(c, 0) for c in [0, 1, 2]):
            break
        rel = r["relative_path"]
        true_label = int(r["label"]) if "label" in r and pd.notna(r["label"]) else -1
        # 이 클래스가 이미 목표를 채웠으면 건너뜀 (불필요한 추론 절약)
        if per_class is not None and true_label >= 0 and got[true_label] >= per_class.get(true_label, 0):
            continue
        try:
            pil = load_image(relative_path=rel)
        except Exception as e:
            print(f"[skip] {rel}: {e}")
            continue
        probs, pred_idx, cam, heatmap, overlay, overlay_thresh, peak = predict_and_cam(pil, thresh=GAME_THRESH)

        # ★ 교육용: 모델이 틀린 이미지는 버린다 (Grad-CAM이 엉뚱할 수 있음)
        if correct_only and pred_idx != true_label:
            skipped_wrong += 1
            continue

        sp = scoring_params(cam, thresh=GAME_THRESH, r_full=GAME_R_FULL)  # 채점용 반지름
        qid = f"{next_id:04d}"
        next_id += 1
        if true_label >= 0:
            got[true_label] += 1

        base_img = pil.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")
        s3.upload_fileobj(_png_bytes(base_img), OUT_BUCKET, f"{OUT_PREFIX}images/{qid}.png")
        # 교육게임용 gradcam은 '핵심영역 강조' 버전을 저장 (보는 사람에게 더 직관적)
        s3.upload_fileobj(_png_bytes(overlay_thresh), OUT_BUCKET, f"{OUT_PREFIX}gradcam/{qid}.png")
        # 전체 overlay(히트맵 전체 합성)도 저장
        s3.upload_fileobj(_png_bytes(overlay), OUT_BUCKET, f"{OUT_PREFIX}overlay/{qid}.png")
        if save_heatmap_too:
            s3.upload_fileobj(_png_bytes(heatmap), OUT_BUCKET, f"{OUT_PREFIX}heatmap/{qid}.png")

        true_label = int(r["label"]) if "label" in r and pd.notna(r["label"]) else -1
        rows.append({
            "id": qid,
            "relative_path": rel,
            "true_label": true_label,
            "true_label_kr": CLASS_NAMES_KR[true_label] if true_label >= 0 else "",
            "pred_label": pred_idx,
            "pred_label_kr": CLASS_NAMES_KR[pred_idx],
            "prob_normal": round(float(probs[0]), 4),
            "prob_bacteria": round(float(probs[1]), 4),
            "prob_virus": round(float(probs[2]), 4),
            "correct": int(true_label == pred_idx) if true_label >= 0 else -1,
            # ---- 게임 채점용 (224 기준 좌표/반지름) ----
            "peak_x": sp["peak_x"],       # 만점존 중심 = CAM 피크
            "peak_y": sp["peak_y"],
            "r_full": sp["r_full"],       # 만점존 반지름 (이 안=100점)
            "r_cam": sp["r_cam"],         # CAM 영역 근사 반지름 (이 밖=급감점)
            "img_size": IMG_SIZE,
            "image_key": f"{OUT_PREFIX}images/{qid}.png",
            "gradcam_key": f"{OUT_PREFIX}gradcam/{qid}.png",
            "overlay_key": f"{OUT_PREFIX}overlay/{qid}.png",
        })
        done = sum(got.values())
        if done % 10 == 0:
            tgt = sum(per_class.values()) if per_class else '전부'
            print(f"  담음 {done}/{tgt}  (정상{got[0]}/세균성{got[1]}/바이러스{got[2]}, 틀려서버림 {skipped_wrong})")

    q_df = pd.DataFrame(rows)
    csv_buf = io.StringIO()
    q_df.to_csv(csv_buf, index=False)
    s3.put_object(Bucket=OUT_BUCKET, Key=f"{OUT_PREFIX}questions.csv",
                  Body=csv_buf.getvalue().encode("utf-8"))
    print(f"\n완료: {len(q_df)}장 저장 (모두 모델이 맞춘 것) → s3://{OUT_BUCKET}/{OUT_PREFIX}")
    if len(q_df):
        print("저장된 클래스 분포:")
        for lab in [0, 1, 2]:
            c = int((q_df['true_label'] == lab).sum())
            tgt = f"/{per_class.get(lab)}" if per_class else ""
            print(f"  {CLASS_NAMES_KR[lab]:12s}: {c}{tgt}장")
    print(f"(참고: 틀려서 제외한 이미지 {skipped_wrong}장)")
    # 목표 미달 경고
    if per_class is not None:
        for lab in [0, 1, 2]:
            if got[lab] < per_class.get(lab, 0):
                print(f"  [주의] {CLASS_NAMES_KR[lab]}: 목표 {per_class[lab]} 중 {got[lab]}장만 (맞춘 게 부족). df를 더 크게 넘기거나 목표를 낮출 것.")
    print("\n[검수용] 로컬 맥 터미널에서 아래 명령으로 한 번에 다운로드:")
    print(f"aws s3 sync s3://{OUT_BUCKET}/{OUT_PREFIX} ./education_game_review --profile aws-say3-11")
    return q_df


print("배치 저장 함수 준비 완료 (아직 실행 안 함). 교육용이라 '맞춘 것만' 저장됨.")
print("사용 예 (CELL 8 참고):")
print('#   test_df = pd.read_csv(CSV_PATH); test_df = test_df[test_df.split=="test"]')
print('#   test_df = test_df.sample(frac=1, random_state=42)   # 섞기')
print('#   q = build_education_game(test_df, per_class={0:34, 1:33, 2:33})  # 맞춘 것만 채움')

## CELL 8 — 교육용 배치 저장 (모델이 '맞춘' 것만, 비율 맞춤)
**순서대로**: ① 데이터 준비 → ② 먼저 30장 검수 → ③ 괜찮으면 100장 본실행.
각 단계는 주석을 풀어 실행. S3에 실제 업로드되므로 의도적으로 돌릴 것.

- 교육용이라 `build_education_game`은 **모델이 맞춘 이미지만** 담는다 (틀린 건 Grad-CAM이 엉뚱할 수 있어 제외).
- `per_class={0:n, 1:n, 2:n}`로 클래스별 목표 개수 지정 → 맞춘 것으로 그 수만큼 채움.
- test 전체(624장)를 **섞어서** 넘긴다. 틀린 게 걸러지므로 df는 넉넉해야 함.
- 저장 후 출력되는 `aws s3 sync ...` 명령을 **로컬 맥 터미널**에 붙여 받아 눈으로 검수.

In [ ]:
# ① 데이터 준비: test 세트를 섞어서 pool 구성 (맞춘 것만 걸러낼 거라 넉넉히)
# pool = pd.read_csv(CSV_PATH)
# pool = pool[pool['split'] == 'test']
# pool = pool.sample(frac=1.0, random_state=42).reset_index(drop=True)  # 섞기
# print('test pool:', len(pool), '장')
# pool['label'].value_counts().sort_index()

In [ ]:
# ② 먼저 30장 검수 (맞춘 것만, 균등 10/10/10). Grad-CAM/threshold/채점 품질 확인용.
# q30 = build_education_game(pool, per_class={0: 10, 1: 10, 2: 10})
# q30.head()
#
#  → 출력된 aws s3 sync 명령을 로컬 터미널에 붙여 받아보고,
#    education_game_review/gradcam/ 폴더를 넘겨보며 '교육용으로 쓸만한지' 확인.

In [ ]:
# ③ 30장이 만족스러우면 100장 본실행 (맞춘 것만, 균등 34/33/33)
#    주의: 바이러스성은 test에 148장뿐 + 오답 제외라, 33장을 못 채우면 경고가 뜬다.
# q100 = build_education_game(pool, per_class={0: 34, 1: 33, 2: 33})
# q100.head()

## CELL 8.5 — (확인) 채점 로직 미리보기
게임 채점식이 어떻게 동작하는지 한 장으로 확인. 3단 구조:
- 만점존(피크에서 `r_full`) 안 → 100점
- CAM 영역 안 → 거리비례 완만 감점 (경계 50점)
- CAM 영역 밖 → 50점에서 급격히 감점

`score_click(클릭x, 클릭y, peak_x, peak_y, r_full, r_cam)` 를 게임이 그대로 쓰면 된다.

In [ ]:
# 샘플 한 장으로 채점 파라미터 뽑고, 가상의 클릭 몇 개에 점수 매겨보기
_rel = "test/PNEUMONIA/person100_bacteria_475.jpeg"
_pil = load_image(relative_path=_rel)
_probs, _pred, _cam, _h, _ov, _ovt, _peak = predict_and_cam(_pil, thresh=GAME_THRESH)
sp = scoring_params(_cam, thresh=GAME_THRESH, r_full=GAME_R_FULL)
print("채점 파라미터:", sp)
px, py, rf, rc = sp["peak_x"], sp["peak_y"], sp["r_full"], sp["r_cam"]

tests = [
    (px, py, "피크 정중앙"),
    (px + int(rf*0.8), py, "만점존 가장자리"),
    (px + int((rf+rc)//2), py, "CAM 영역 중간"),
    (px + int(rc), py, "CAM 경계"),
    (px + int(rc*1.8), py, "영역 밖(멀리)"),
    (10, 10, "완전 엉뚱(좌상단)"),
]
for cx, cy, label in tests:
    s = score_click(cx, cy, px, py, rf, rc)
    print(f"  클릭({cx:3d},{cy:3d}) {label:16s} → {s:5.1f}점")

# 피크 위치를 초록 X로 표시해 '어디가 정답인지' 확인 (검수용)
plt.figure(figsize=(5,5))
plt.imshow(_ovt)
plt.plot(px, py, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.title(f"정답 피크=({px},{py}), r_full={rf}, r_cam={rc}")
plt.axis('off'); plt.show()

## CELL 9 — 진단: Grad-CAM이 코드 문제인지 모델 특성인지 가르기
"어깨/상단 쏠림"의 원인을 확인한다.
1. **모델 로드 검증**: 샘플 몇 장 예측이 정답과 맞는지 (로드가 제대로면 정확도 높아야 함)
2. **CAM 무게중심**: 히트맵이 평균적으로 이미지의 어느 높이에 쏠리는지 수치로 (0=맨위, 1=맨아래)
3. **라이브러리 교차검증(선택)**: pytorch-grad-cam이 설치돼 있으면 같은 이미지로 비교

In [ ]:
# --- 1) 모델 로드 검증: 샘플 예측이 정답과 맞나? ---
# (로드가 잘못됐으면 예측이 엉망 → CAM도 의미 없음. 먼저 이걸 확인)
check = [
    ("test/NORMAL/IM-0001-0001.jpeg", 0),
    ("test/NORMAL/IM-0003-0001.jpeg", 0),
    ("test/PNEUMONIA/person100_bacteria_475.jpeg", 1),
    ("test/PNEUMONIA/person100_bacteria_477.jpeg", 1),
]
ok = 0
for rel, true in check:
    try:
        pil = load_image(relative_path=rel)
        x = preprocess(pil)
        with torch.no_grad():
            p = F.softmax(model(x), dim=1)[0].cpu().numpy()
        pred = int(p.argmax())
        hit = (pred == true)
        ok += hit
        print(f"{'OK ' if hit else 'X  '} {rel.split('/')[-1]:40s} 정답={CLASS_NAMES[true]:20s} 예측={CLASS_NAMES[pred]:20s} ({p[pred]*100:.0f}%)")
    except Exception as e:
        print(f"[skip] {rel}: {e}")
print(f"\n→ {ok}/{len(check)} 맞음. (대부분 맞으면 모델 로드 정상 = CAM 쏠림은 코드버그 아님)")

In [ ]:
# --- 2) CAM 무게중심: 히트맵이 평균적으로 어느 높이에 쏠리나 ---
# y_center: 0=맨위(어깨/목), 0.5=가운데(폐), 1=맨아래(횡격막)
import numpy as np
ys = []
for rel, _ in check:
    try:
        pil = load_image(relative_path=rel)
        _, _, cam, _, _, _, _ = predict_and_cam(pil)
        grid_y = np.linspace(0, 1, cam.shape[0])[:, None]
        w = cam / (cam.sum() + 1e-8)
        y_center = float((w.sum(axis=1) * grid_y[:, 0]).sum())
        ys.append(y_center)
        print(f"{rel.split('/')[-1]:40s} CAM 세로 무게중심 y={y_center:.2f}")
    except Exception as e:
        print(f"[skip] {rel}: {e}")
if ys:
    m = sum(ys)/len(ys)
    print(f"\n평균 y={m:.2f}  → 0.2 이하면 상단(어깨)쏠림, 0.4~0.6이면 폐 중심")

In [ ]:
# --- 3) (선택) pytorch-grad-cam 라이브러리로 교차검증 ---
# 우리 직접구현 CAM과 라이브러리 CAM이 같은 곳을 짚으면 → 구현은 정상, 쏠림은 모델 특성.
# 설치돼 있을 때만 동작 (SageMaker에 없으면: %pip install grad-cam)
try:
    from pytorch_grad_cam import GradCAM as LibGradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

    rel = check[2][0]  # 세균성 샘플
    pil = load_image(relative_path=rel)
    x = preprocess(pil)
    with torch.no_grad():
        pred = int(model(x).argmax(1).item())

    lib = LibGradCAM(model=model, target_layers=[model.layer4[-1]])
    lib_cam = lib(input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]

    _, _, our_cam, _, _, _, _ = predict_and_cam(pil, class_idx=pred)

    fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
    ax[0].imshow(pil.resize((IMG_SIZE, IMG_SIZE)), cmap="gray"); ax[0].set_title("원본")
    ax[1].imshow(our_cam, cmap="jet"); ax[1].set_title("우리 직접구현 CAM")
    ax[2].imshow(lib_cam, cmap="jet"); ax[2].set_title("pytorch-grad-cam")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    print("두 CAM이 비슷하면 → 구현 정상. 쏠림은 모델이 실제로 그 영역을 보는 것.")
except ImportError:
    print("pytorch-grad-cam 미설치. 교차검증 건너뜀. (원하면: %pip install grad-cam 후 재실행)")